# Capítulo 6 — Otimização de índices (Index Optimization)

**Livro:** começa na página **308** do PDF.

📗 **Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # ia.py (modelos via OpenRouter) e corpus.py (base de conhecimento de exemplo)
from ia import *
from corpus import DOCS
import re, json, math, random
import numpy as np
print("Pronto. Modelo padrão:", MODELO, "| documentos na base:", len(DOCS))

## From small to large node-sentence sliding window and parent-child text chunks · PDF p. 309

#### 🧪 Exercício: Indexar frases, devolver a janela

Divida todos os textos em frases, indexe cada frase (BM25) e, ao encontrar a melhor frase, devolva também a frase anterior e a seguinte do mesmo documento (*sentence window*).

In [ ]:
from rank_bm25 import BM25Okapi
frases = [(d["id"], i, f) for d in DOCS for i, f in enumerate(re.split(r"(?<=\.)\s+", d["texto"]))]
bm = BM25Okapi([re.findall(r"\w+", f.lower()) for _, _, f in frases])
k = int(np.argmax(bm.get_scores("garrafa de água".split())))
doc, i, _ = frases[k]
janela = [f for d, j, f in frases if d == doc and abs(j - i) <= 1]
print(" ".join(janela))

## Hybrid retrieval improving retrieval accuracy and expanding coverage · PDF p. 336

#### 🧪 Exercício: Busca híbrida com RRF

Combine o ranking do BM25 com o ranking semântico (embeddings) usando *Reciprocal Rank Fusion*: score = Σ 1/(60 + posição). Mostre o top-3 de cada método e o fundido para "onde estudar com notebook?".

In [ ]:
from rank_bm25 import BM25Okapi
q = "onde estudar com notebook?"
bm = BM25Okapi([re.findall(r"\w+", d["texto"].lower()) for d in DOCS]).get_scores(re.findall(r"\w+", q.lower()))
sem = similaridade(embeddings([d["texto"] for d in DOCS]), embeddings(q)[0])
rk = {nome: list(np.argsort(-s)) for nome, s in [("bm25", bm), ("semantico", sem)]}
rrf = {i: sum(1 / (60 + r.index(i) + 1) for r in rk.values()) for i in range(len(DOCS))}
for nome, r in rk.items(): print(nome, [DOCS[i]["titulo"] for i in r[:3]])
print("RRF", [DOCS[i]["titulo"] for i in sorted(rrf, key=rrf.get, reverse=True)[:3]])